# TaqSim starter: one river, one reservoir, one farm

This notebook runs in **Google Colab** with a free Google account. Nothing is installed on your laptop.

What you will do, in three steps:

1. **Install TaqSim** (about one minute) and build a small water system of the Zarafshan type: a river, a headworks that
   diverts water into a canal, a reservoir on the canal, an irrigation district, and the river continuing downstream.
2. **Run it** for three years and look at what happens: how full the reservoir is, and whether the farm gets its water.
3. **Hand the work to an LLM**: ask Gemini (built into Colab) or any free chat model to change the model for you.

Run the cells in order (`Runtime → Run all` is fine). If a cell fails, read the error, then scroll down to the section
*Hand the work to an LLM* for how to get it fixed.

In [ ]:
# @title Code
# Step 1: install TaqSim v0.1.4 (pure Python, no compiler needed). About a minute in Colab.
# Colab already has numpy, pandas, matplotlib and friends. We add only the two packages it lacks, then TaqSim
# itself without its dependency list, so that Colab's own Jupyter packages stay untouched.
%pip install -q deap ctrl-freak
%pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"

import sys
import taqsim

print("TaqSim", taqsim.__version__, "on Python", sys.version.split()[0])

## The system

```
river  ──▶  headworks  ──▶  river_downstream        (what is not diverted stays in the river)
               │
               └──▶  reservoir  ──▶  farm  ──▶  drain
```

- **river**: monthly inflow at Ravatkhoja, in million cubic metres (Mm³) per month. The values are the
  median of each calendar month over 2010 to 2023, computed from the daily gauge record used in the course.
- **headworks**: a splitter. A fixed *share* of the river goes into the canal that feeds the reservoir; the rest flows on.
- **reservoir**: capacity 667 Mm³, of which 66.7 Mm³ is dead storage (below the lowest outlet). It starts the
  hydrological year on 1 October with 170 Mm³. These are the values used in the course game for Kattakurgan.
- **farm**: the Kattakurgan irrigation district, needing 385 Mm³ a year, almost all of it from April to September.
- **drain** and **river_downstream**: where water leaves the model.

Everything is **per time step**: one step is one month, and all numbers are Mm³ per month. The reservoir follows one
simple rule: *release what the farm needs this month, as long as there is water above dead storage*.

The year runs October to September (the hydrological year), so index 0 is October.

In [ ]:
# @title Code
MONTHS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

# River inflow at Ravatkhoja, Mm3 per month, median 2010-2023, October first.
INFLOW_MM3 = (203.0, 162.9, 133.5, 124.3, 106.2, 124.2, 155.6, 381.5, 771.1, 993.4, 711.9, 350.5)

# Irrigation need of the Kattakurgan district, Mm3 per month, October first.
FARM_NEED_MM3 = (18.8, 0.0, 0.0, 0.0, 0.0, 3.3, 18.5, 38.6, 43.0, 83.0, 113.4, 66.8)

YEARS = 3
CANAL_SHARE = 0.08  # share of the river diverted into the canal at the headworks

print(f"River brings {sum(INFLOW_MM3):,.0f} Mm3 a year; the farm needs {sum(FARM_NEED_MM3):,.0f} Mm3 a year.")

## Two rules

TaqSim nodes do not decide anything by themselves. You give them a *rule* (a small Python class):

- the headworks needs a **split policy**: how much goes where;
- the reservoir needs a **release policy**: how much to let out each month.

A rule is a frozen dataclass with one method. Fields listed in `__params__` are the knobs an optimiser may turn later.

In [ ]:
# @title Code
from dataclasses import dataclass
from typing import ClassVar

from taqsim import Splitter, Storage, Strategy, Timestep


@dataclass(frozen=True)
class FixedShare(Strategy):
    '''Send a fixed share of the river into the canal; the rest stays in the river.'''

    __params__: ClassVar[tuple[str, ...]] = ("share",)
    __bounds__: ClassVar[dict[str, tuple[float, float]]] = {"share": (0.0, 1.0)}
    share: float = 0.1

    def split(self, node: Splitter, amount: float, t: Timestep) -> dict[str, float]:
        return {"reservoir": amount * self.share, "river_downstream": amount * (1.0 - self.share)}


@dataclass(frozen=True)
class ReleaseTheNeed(Strategy):
    '''Release what the farm needs this month, never more than the water above dead storage.'''

    __params__: ClassVar[tuple[str, ...]] = ()

    def release(self, node: Storage, inflow: float, t: Timestep) -> float:
        need = FARM_NEED_MM3[t.index % 12]
        return min(need, node.storage - node.dead_storage)

## Build the system

Nodes only process water. Edges say who sends water to whom. `validate()` checks the network is complete.

In [ ]:
# @title Code
from taqsim import Demand, Edge, Frequency, Sink, Source, TimeSeries, WaterSystem
from taqsim.node import NoLoss


def build_system(canal_share: float = CANAL_SHARE, years: int = YEARS) -> WaterSystem:
    # location=(lat, lon) is meant for real map coordinates; here it only places the nodes in the drawing below.
    system = WaterSystem(frequency=Frequency.MONTHLY)
    system.add_node(Source(id="river", inflow=TimeSeries(values=list(INFLOW_MM3) * years), location=(1.0, 0.0)))
    system.add_node(Splitter(id="headworks", split_policy=FixedShare(share=canal_share), location=(1.0, 1.0)))
    system.add_node(
        Storage(
            id="reservoir",
            capacity=667.0,
            dead_storage=66.7,
            initial_storage=170.0,
            release_policy=ReleaseTheNeed(),
            loss_rule=NoLoss(),  # no evaporation yet: exercise 2 below
            location=(0.4, 2.0),
        )
    )
    system.add_node(Demand(id="farm", requirement=TimeSeries(values=list(FARM_NEED_MM3) * years), location=(0.4, 3.0)))
    system.add_node(Sink(id="drain", location=(0.4, 4.0)))
    system.add_node(Sink(id="river_downstream", location=(1.6, 3.0)))
    for source, target in (
        ("river", "headworks"),
        ("headworks", "reservoir"),
        ("headworks", "river_downstream"),
        ("reservoir", "farm"),
        ("farm", "drain"),
    ):
        system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))
    system.validate()
    return system


system = build_system()
fig, ax = system.visualize(figsize=(8, 3.5), title="The starter system")

## Run it and read the result

A run records *events* at every node: water received, stored, released, spilled, and any deficit at the farm.
Reading a result means summing the events you care about, month by month.

In [ ]:
# @title Code
import pandas as pd
from taqsim.node import DeficitRecorded, WaterLost, WaterReleased, WaterSpilled, WaterStored


def monthly(node, event_type, field: str = "amount", steps: int = 12 * YEARS) -> list[float]:
    totals = [0.0] * steps
    for event in node.events_of_type(event_type):
        totals[event.t] += getattr(event, field)
    return totals


def run(canal_share: float = CANAL_SHARE) -> pd.DataFrame:
    system = build_system(canal_share)
    system.simulate(12 * YEARS)
    reservoir, farm = system.nodes["reservoir"], system.nodes["farm"]
    stored = monthly(reservoir, WaterStored)
    lost = monthly(reservoir, WaterLost)
    released = monthly(reservoir, WaterReleased)
    storage, level = [], reservoir.initial_storage
    for s, l, r in zip(stored, lost, released):
        level += s - l - r
        storage.append(level)
    return pd.DataFrame(
        {
            "year": [i // 12 + 1 for i in range(12 * YEARS)],
            "month": MONTHS * YEARS,
            "river_in": list(INFLOW_MM3) * YEARS,
            "to_canal": [v * canal_share for v in INFLOW_MM3] * YEARS,
            "spilled": monthly(reservoir, WaterSpilled),
            "storage_end": storage,
            "farm_need": list(FARM_NEED_MM3) * YEARS,
            "shortage": monthly(farm, DeficitRecorded, "deficit"),
        }
    ).round(1)


result = run()
print(f"Total shortage over {YEARS} years: {result.shortage.sum():.1f} Mm3; "
      f"reservoir ends with {result.storage_end.iloc[-1]:.1f} Mm3.")
result[result.year == YEARS]

In [ ]:
# @title Code
import matplotlib.pyplot as plt

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
top.plot(result.storage_end, color="tab:blue")
top.axhline(66.7, color="grey", ls="--", lw=1)
top.set_ylabel("reservoir, Mm³")
top.set_title(f"Canal share {CANAL_SHARE:.0%}: the reservoir drains over three years")
bottom.bar(result.index, result.farm_need, color="lightgrey", label="farm need")
bottom.bar(result.index, result.shortage, color="tab:red", label="shortage")
bottom.set_ylabel("Mm³ per month")
bottom.set_xticks(range(0, 12 * YEARS, 3))
bottom.set_xticklabels([f"{MONTHS[i % 12]} y{i // 12 + 1}" for i in range(0, 12 * YEARS, 3)])
bottom.legend(loc="upper left")
plt.tight_layout()

## Hand the work to an LLM

No LLM has seen TaqSim before. It can still write correct TaqSim code **if the notebook already contains working
examples and the cheat sheet below**. That is why this notebook is long: it is the LLM's reference.

**In Colab**: click the blue **Gemini** button at the bottom centre of the page (or the sparkle icon in a code cell), and
ask in plain words. Gemini sees the cells above. Insert its code as a new cell, run it, and if it fails, paste the error back.

Try these, one at a time:

1. *"Find the smallest canal share, in steps of 1 percent, for which the farm has no shortage in any of the three years.
   Use the run() function."*
2. *"Add evaporation to the reservoir: each month the reservoir loses a share of its content. Write a loss rule class
   like NoLoss, with the monthly shares October-first: 1.6, 1.0, 0.6, 0.6, 0.6, 1.2, 2.1, 4.4, 6.6, 7.6, 6.2, 4.1 percent.
   Rerun and show me how much more shortage that causes."*
3. *"Add a city between the reservoir and the farm. It needs 6 Mm³ every month, is served before the farm, and returns
   70 percent of its water to the river. Use a Demand node with consumption_fraction."*
4. *"Replace the typical year by a dry year: multiply the inflow by 0.85. What canal share does the farm need now?"*

Then ask your own question. Good prompts say **what the water should do**, not what code to write.

**Outside Colab** (Gemini app, ChatGPT Free, claude.ai Free): paste the cheat sheet below and the cell you want to change
into the chat, ask for the new cell, paste it back here and run it. Same loop: context in, code out, error back.

## TaqSim cheat sheet (v0.1.4)

**Imports**
```python
from taqsim import (Demand, Edge, Frequency, PassThrough, Sink, Source, Splitter, Storage,
                    Strategy, TimeSeries, Timestep, WaterSystem)
from taqsim.node import (NoLoss, DeficitRecorded, WaterReceived, WaterReleased, WaterSpilled,
                         WaterStored, WaterLost, WaterConsumed, WaterDistributed, EVAPORATION, SEEPAGE)
```

**Nodes** (each needs a unique `id` and may take `location=(lat, lon)`; nodes never name their targets, edges do):

| Node | Required arguments | What it does |
|---|---|---|
| `Source(id, inflow=TimeSeries)` | inflow per step | puts water in |
| `Splitter(id, split_policy=...)` | a split policy | sends shares to several targets |
| `Storage(id, capacity, release_policy=..., loss_rule=..., initial_storage=0, dead_storage=0)` | capacity, both rules | reservoir; spills above capacity |
| `Demand(id, requirement=TimeSeries, consumption_fraction=1.0, efficiency=1.0)` | requirement per step | takes what it needs, passes the rest on |
| `PassThrough(id, capacity=None)` | nothing | junction or turbine, spills above capacity |
| `Sink(id)` | nothing | end of the system |

**Edges**: `system.add_edge(Edge(id="a_to_b", source="a", target="b"))`. Every node except a Sink must have exactly one
outgoing edge, except a Splitter, which has one per target. Every node must reach a Sink.

**System**: `system = WaterSystem(frequency=Frequency.MONTHLY)` (or `Frequency.DAILY`), then `add_node`, `add_edge`,
`validate()`, `simulate(n_steps)`. All values are per time step; TaqSim does not convert units. `TimeSeries(values=[...])`
needs one value per step, none negative. To rerun with different settings, build a new system (`system.reset()` also works).

**Rules** are frozen dataclasses with one method; `t.index` is the step number (0, 1, 2, ...):
```python
@dataclass(frozen=True)
class MyRelease(Strategy):                      # Storage.release_policy
    __params__: ClassVar[tuple[str, ...]] = ("rate",)
    rate: float = 10.0
    def release(self, node: Storage, inflow: float, t: Timestep) -> float:
        return min(self.rate, node.storage - node.dead_storage)

@dataclass(frozen=True)
class MySplit(Strategy):                        # Splitter.split_policy, keys are target node ids
    __params__: ClassVar[tuple[str, ...]] = ("share",)
    share: float = 0.5
    def split(self, node: Splitter, amount: float, t: Timestep) -> dict[str, float]:
        return {"left": amount * self.share, "right": amount * (1 - self.share)}

@dataclass(frozen=True)
class MyLoss:                                   # Storage.loss_rule: NOT a Strategy, returns {reason: volume}
    share: float = 0.02
    def calculate(self, node: Storage, t: Timestep) -> dict:
        return {EVAPORATION: node.storage * self.share}
```

**Reading results**: every node keeps events. `node.events_of_type(WaterStored)` returns a list; each event has
`.amount` and `.t` (`DeficitRecorded` has `.required`, `.actual`, `.deficit`). Sum them by `t` to get a monthly series.
`system.nodes["reservoir"].storage` is the content after the run. `system.visualize()` draws the network.

**Order of operations in a Storage each step**: receive, store (spill if above capacity), lose, release.
A Demand takes `min(received, requirement)`, consumes `consumption_fraction` of it and passes the remainder plus any excess
downstream. Water below `dead_storage` cannot be released but can still evaporate.

**Common mistakes**: forgetting an edge to a Sink (validate fails); a split policy that names a target that has no edge;
negative or NaN values in a TimeSeries; mixing units (m³/s in one place, Mm³ per month in another); using `trace[t]`
instead of `.get(t, 0.0)` on objective traces.

## Full documentation for a longer chat

TaqSim ships its own documentation inside the package. The next cell zips it so you can download it and upload it to any
chat that accepts files (Gemini, ChatGPT and Claude all do). It also prints the first guide so Gemini in Colab sees it.

In [ ]:
# @title Code
import shutil
from pathlib import Path
from taqsim import get_docs_path

docs = Path(get_docs_path())
archive = shutil.make_archive("taqsim_docs", "zip", docs)
files = sorted(p.relative_to(docs) for p in docs.rglob("*.md"))
print(f"{len(files)} documentation files zipped into {archive}:")
print("\n".join(f"  {p}" for p in files))

try:
    from google.colab import files as colab_files  # only exists inside Colab

    colab_files.download(archive)
except ImportError:
    pass

## If you have Python on your laptop

The same install line works anywhere with Python 3.12 or newer and `git`:

```
pip install "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
```

Then any open-source coding agent can work on this notebook with a free model behind it, for example OpenCode with one
of its free hosted models, or Codex CLI signed in with a free ChatGPT account. Put the cheat sheet above into a file
called `AGENTS.md` in the folder: that is how these tools pick up context about a library they do not know.